# RAG vs. No-RAG — Multi-Family LLM Evaluation
## Notebook 04 — Evaluation

Evaluates **22 models across 5 families** under **5 conditions**:

| Condition | RAG corpus | Embedding | Language |
|-----------|-----------|-----------|----------|
| `no_rag`        | — | — | — |
| `rag_a`         | Corpus A (Župan Q&A) | nomic-embed-text | English |
| `rag_b_nomic`   | Corpus B (RRiF PDF) | nomic-embed-text | Croatian ← cross-lingual gap |
| `rag_b_qwen3s`  | Corpus B (RRiF PDF) | qwen3-embedding:0.6b | Croatian |
| `rag_b_qwen3m`  | Corpus B (RRiF PDF) | qwen3-embedding:4b | Croatian |

**Total API calls:** 22 × 5 × 1,060 = **116,600**

Checkpoints are saved after every model+condition pair.
Re-running safely skips completed pairs.


---
## ⚙️ Configuration

In [4]:
# ── CONFIGURATION — edit only this cell ──────────────────────────────────────

OLLAMA_BASE_URL = "http://100.119.36.113:11434"   # ← set your server IP

DATA_DIR    = "data"
CHROMA_DIR  = f"{DATA_DIR}/chroma_db"
RESULTS_DIR = f"{DATA_DIR}/results"
FIGURES_DIR = f"{DATA_DIR}/figures"

TOP_K       = 5
TEMPERATURE = 0
MAX_TOKENS  = 256

# ── Embedding models ───────────────────────────────────────────────────────────
EMBED_MODELS = {
    "nomic":   "nomic-embed-text",       # English-optimised
    "qwen3s":  "qwen3-embedding:0.6b",   # Multilingual lightweight
    "qwen3m":  "qwen3-embedding:4b",     # Multilingual mid-size
}

# ── ChromaDB collections ───────────────────────────────────────────────────────
COLLECTIONS = {
    "rag_a":        ("corpus_a_nomic",    "nomic"),
    "rag_b_nomic":  ("corpus_b_nomic",    "nomic"),
    "rag_b_qwen3s": ("corpus_b_qwen3s",   "qwen3s"),
    "rag_b_qwen3m": ("corpus_b_qwen3m",   "qwen3m"),
}
# Format: condition → (collection_name, embed_key)

CONDITIONS = ["no_rag", "rag_a", "rag_b_nomic", "rag_b_qwen3s", "rag_b_qwen3m"]

COND_LABELS = {
    "no_rag":        "No RAG",
    "rag_a":         "RAG-A (EN, nomic)",
    "rag_b_nomic":   "RAG-B (HR, nomic)",
    "rag_b_qwen3s":  "RAG-B (HR, qwen3-0.6b)",
    "rag_b_qwen3m":  "RAG-B (HR, qwen3-4b)",
}

# ── Model families ─────────────────────────────────────────────────────────────
MODELS = {
    "Qwen3.5": [
        "qwen3.5:0.8b", "qwen3.5:2b", "qwen3.5:4b", "qwen3.5:9b",
        "qwen3.5:27b",  "qwen3.5:35b", "qwen3.5:122b",
    ],
    "Llama": [
        "llama3.2:1b", "llama3.2:3b", "llama3.1:8b", "llama3.3:70b",
    ],
    "Gemma": [
        "gemma3:1b", "gemma3:4b", "gemma3:12b", "gemma3:27b",
    ],
    "Phi": [
        "phi4-mini:latest", "phi4:latest",
    ],
    "DeepSeek": [
        "deepseek-r1:1.5b", "deepseek-r1:7b", "deepseek-r1:14b",
        "deepseek-r1:32b",  "deepseek-r1:70b",
    ],
}
ALL_MODELS = [m for family in MODELS.values() for m in family]

MODEL_PARAMS = {
    "qwen3.5:0.8b":0.8,  "qwen3.5:2b":2,    "qwen3.5:4b":4,
    "qwen3.5:9b":9,      "qwen3.5:27b":27,   "qwen3.5:35b":35,
    "qwen3.5:122b":122,
    "llama3.2:1b":1,     "llama3.2:3b":3,    "llama3.1:8b":8,
    "llama3.3:70b":70,
    "gemma3:1b":1,       "gemma3:4b":4,      "gemma3:12b":12,
    "gemma3:27b":27,
    "phi4-mini:latest":3.8,     "phi4:latest":14,
    "deepseek-r1:1.5b":1.5, "deepseek-r1:7b":7,  "deepseek-r1:14b":14,
    "deepseek-r1:32b":32,   "deepseek-r1:70b":70,
}

import os
for d in [DATA_DIR, CHROMA_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

total_calls = len(ALL_MODELS) * len(CONDITIONS) * 1060
print("✓ Configuration loaded")
print(f"  Models:      {len(ALL_MODELS)}  ({', '.join(MODELS.keys())})")
print(f"  Conditions:  {len(CONDITIONS)}")
print(f"  Questions:   1,060")
print(f"  Total calls: {total_calls:,}")
print()
print("  Embedding models:")
for k, v in EMBED_MODELS.items():
    print(f"    {k:8s}: {v}")


✓ Configuration loaded
  Models:      22  (Qwen3.5, Llama, Gemma, Phi, DeepSeek)
  Conditions:  5
  Questions:   1,060
  Total calls: 116,600

  Embedding models:
    nomic   : nomic-embed-text
    qwen3s  : qwen3-embedding:0.6b
    qwen3m  : qwen3-embedding:4b


---
## 📦 Dependencies

In [5]:
import subprocess
subprocess.run(
    ["pip", "install", "-q", "--root-user-action=ignore",
     "chromadb", "requests", "tqdm"],
    capture_output=True
)
print("✓ OK")


✓ OK


---
## 📥 Load data & ChromaDB

In [6]:
import json, time, re, os, requests
import chromadb
from tqdm import tqdm
from datetime import datetime

with open(f"{DATA_DIR}/test_set.json") as f:
    test_set = json.load(f)

lookup_qs   = [q for q in test_set if q["q_type"] == "lookup"]
describe_qs = [q for q in test_set if q["q_type"] == "describe"]
print(f"✓ Test set: {len(test_set)} questions "
      f"({len(lookup_qs)} lookup, {len(describe_qs)} describe)")

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
loaded_colls  = {}
for cond, (coll_name, _) in COLLECTIONS.items():
    try:
        loaded_colls[cond] = chroma_client.get_collection(coll_name)
    except Exception:
        loaded_colls[cond] = None

print("\nChromaDB collections:")
for cond, coll in loaded_colls.items():
    status = f"{coll.count()} docs" if coll else "MISSING"
    print(f"  {cond:20s}: {status}")


✓ Test set: 1060 questions (530 lookup, 530 describe)

ChromaDB collections:
  rag_a               : 522 docs
  rag_b_nomic         : 2462 docs
  rag_b_qwen3s        : 2462 docs
  rag_b_qwen3m        : 2462 docs


In [7]:
EMBED_CACHE_PATH = f"{DATA_DIR}/embedding_cache.json"

def load_embed_cache():
    if os.path.exists(EMBED_CACHE_PATH):
        with open(EMBED_CACHE_PATH) as f:
            return json.load(f)
    return {}

def save_embed_cache(cache):
    with open(EMBED_CACHE_PATH, "w") as f:
        json.dump(cache, f)

embed_cache = load_embed_cache()
print(f"✓ Embedding cache: {len(embed_cache)} upita")

✓ Embedding cache: 3100 upita


---
## 🔧 Helper functions

In [8]:
import json
# Valid accounts for hallucination detection
'''with open("accounts.jsonl") as f:
    _raw = [json.loads(l) for l in f if l.strip()]
VALID_ACCOUNTS = set()
for d in _raw:
    a = json.loads(next(c for c in d["conversations"]
                        if c["role"] == "assistant")["content"])
    VALID_ACCOUNTS.add(a.get("account_number", ""))'''
# Valid accounts for hallucination detection
# Prošireni skup: accounts.jsonl + sva konta iz Corpus B (RRiF PDF)

# 1. Iz Županovog dataseta (522 konta)
with open("accounts.jsonl") as f:
    _raw = [json.loads(l) for l in f if l.strip()]

VALID_ACCOUNTS = set()
for d in _raw:
    a = json.loads(next(c for c in d["conversations"]
                        if c["role"] == "assistant")["content"])
    VALID_ACCOUNTS.add(a.get("account_number", ""))

print(f"  Konta iz accounts.jsonl:    {len(VALID_ACCOUNTS)}")

# 2. Iz Corpus B — svi RRiF konti (2,462 konta)
with open("data/corpus_b_rrif_documents.json", encoding="utf-8") as f:
    corpus_b = json.load(f)

for doc in corpus_b:
    VALID_ACCOUNTS.add(doc["kljuc"])

# Ukloni prazne stringove ako postoje
VALID_ACCOUNTS.discard("")

print(f"  Konta iz Corpus B (RRiF):   {len([d for d in corpus_b])}")
print(f"  Ukupno valid konta:         {len(VALID_ACCOUNTS)}")
print(f"  Valid accounts cache:       {len(VALID_ACCOUNTS)}")


def get_embedding(text, embed_key):
    model = EMBED_MODELS[embed_key]
    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=30
    )
    r.raise_for_status()
    return r.json()["embedding"]


def retrieve(query, condition, k=TOP_K):
    if condition == "no_rag":
        return []
    coll_name, embed_key = COLLECTIONS[condition]
    coll = loaded_colls.get(condition)
    if coll is None:
        return []
    vec = get_embedding_cached(query, embed_key)
    res = coll.query(query_embeddings=[vec], n_results=k,
                     include=["documents","metadatas","distances"])
    return [{"text":       d,
             "metadata":   m,
             "similarity": round(1-dist, 4)}
            for d, m, dist in zip(res["documents"][0],
                                  res["metadatas"][0],
                                  res["distances"][0])]


def build_prompt(question, retrieved_docs, condition):
    if condition == "no_rag" or not retrieved_docs:
        return (
            "You are an accounting expert specializing in the Croatian "
            "Chart of Accounts (Hrvatski računski plan).\n\n"
            f"QUESTION:\n{question}\n\n"
            "Answer concisely and accurately."
        )
    context = "\n\n".join(
        f"[{i+1}] {d['text']}" for i, d in enumerate(retrieved_docs)
    )
    return (
        "You are an accounting expert specializing in the Croatian "
        "Chart of Accounts (Hrvatski računski plan).\n\n"
        "Use only the following context to answer the question accurately.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION:\n{question}\n\n"
        "Answer concisely and accurately based only on the context above."
    )


def strip_thinking(text):
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return cleaned if cleaned else text


def query_model(model, prompt, q_type="lookup"):
    max_tok = 64 if q_type == "lookup" else 256
    t0 = time.time()
    r  = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model":    model,
            "messages": [{"role": "user", "content": prompt}],
            "stream":   False,
            "think":    False,
            "options":  {"temperature": TEMPERATURE, "num_predict": max_tok},
        },
        timeout=300,
    )
    r.raise_for_status()
    elapsed  = time.time() - t0
    response = strip_thinking(r.json()["message"]["content"].strip())
    return response, elapsed


def exact_match(response, ground_truth):
    if not response: return False
    if isinstance(ground_truth, str):
        try: ground_truth = json.loads(ground_truth)
        except: return False
    if not isinstance(ground_truth, dict): return False
    expected = str(ground_truth.get("account_number","")).strip()
    if not expected: return False
    found = re.findall(r"(?<![\d])\b(\d{3,4})\b(?![\d])", response)
    return expected in found


def hallucination_check(response):
    found = re.findall(r"(?<![\d])\b(\d{3,4})\b(?![\d])", response)
    return any(n not in VALID_ACCOUNTS for n in found) if found else False


def checkpoint_path(model, condition):
    safe = model.replace(":","_").replace(".","_")
    return f"{RESULTS_DIR}/{safe}_{condition}.json"

def checkpoint_exists(model, condition):
    return os.path.exists(checkpoint_path(model, condition))

def save_checkpoint(model, condition, results):
    with open(checkpoint_path(model, condition), "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

def load_checkpoint(model, condition):
    with open(checkpoint_path(model, condition)) as f:
        return json.load(f)

def get_embedding_cached(text, embed_key):
    cache_key = f"{embed_key}|{text}"
    if cache_key not in embed_cache:
        embed_cache[cache_key] = get_embedding(text, embed_key)
        if len(embed_cache) % 100 == 0:
            save_embed_cache(embed_cache)
    return embed_cache[cache_key]


print("✓ Helper functions defined")
print(f"  Valid accounts: {len(VALID_ACCOUNTS)}")


  Konta iz accounts.jsonl:    522
  Konta iz Corpus B (RRiF):   2462
  Ukupno valid konta:         2464
  Valid accounts cache:       2464
✓ Helper functions defined
  Valid accounts: 2464


---
## 🔌 Check model availability

In [ ]:
import requests
r = requests.get("http://100.119.36.113:11434/api/tags")
models = [m["name"] for m in r.json()["models"]]
for m in sorted(models):
    if "phi" in m.lower():
        print(repr(m))

In [8]:
r         = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
available = [m["name"] for m in r.json().get("models", [])]

print("Model availability:")
all_ok = True
for family, models in MODELS.items():
    print(f"\n  {family.upper()}")
    for model in models:
        ok     = model in available
        status = "✓" if ok else "✗ MISSING"
        if not ok: all_ok = False
        print(f"    {status}  {model}")

print()
print("✓ All models available" if all_ok else "⚠ Some models missing")


Model availability:

  QWEN3.5
    ✓  qwen3.5:0.8b
    ✓  qwen3.5:2b
    ✓  qwen3.5:4b
    ✓  qwen3.5:9b
    ✓  qwen3.5:27b
    ✓  qwen3.5:35b
    ✓  qwen3.5:122b

  LLAMA
    ✓  llama3.2:1b
    ✓  llama3.2:3b
    ✓  llama3.1:8b
    ✓  llama3.3:70b

  GEMMA
    ✓  gemma3:1b
    ✓  gemma3:4b
    ✓  gemma3:12b
    ✓  gemma3:27b

  PHI
    ✓  phi4-mini:latest
    ✓  phi4:latest

  DEEPSEEK
    ✓  deepseek-r1:1.5b
    ✓  deepseek-r1:7b
    ✓  deepseek-r1:14b
    ✓  deepseek-r1:32b
    ✓  deepseek-r1:70b

✓ All models available


In [9]:
import inspect
print(inspect.getsource(query_model))

def query_model(model, prompt, q_type="lookup"):
    max_tok = 64 if q_type == "lookup" else 256
    t0 = time.time()
    r  = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model":    model,
            "messages": [{"role": "user", "content": prompt}],
            "stream":   False,
            "think":    False,
            "options":  {"temperature": TEMPERATURE, "num_predict": max_tok},
        },
        timeout=300,
    )
    r.raise_for_status()
    elapsed  = time.time() - t0
    response = strip_thinking(r.json()["message"]["content"].strip())
    return response, elapsed



---
## 🧪 Single-question smoke test

In [10]:
test_q    = test_set[0]
model     = ALL_MODELS[0]
condition = "rag_a"

retrieved = retrieve(test_q["question"], condition)
prompt    = build_prompt(test_q["question"], retrieved, condition)

print(f"Model:           {model}")
print(f"Question:        {test_q['question'][:80]}")
print(f"GT:              {test_q['answer_raw'][:80]}")
print(f"Duljina prompta: {len(prompt)} znakova")
print(f"Procjena tokena: {len(prompt)//4}")
print()

response, t = query_model(model, prompt, q_type="lookup")

print(f"Response:        {response[:150]}")
print(f"Latency:         {t:.2f}s")
print(f"EM:              {exact_match(response, test_q['answer'])}")
print(f"Hallucin.:       {hallucination_check(response)}")
print(f"Retrieved:       {[r['metadata'].get('account_number', r['metadata'].get('kljuc','')) for r in retrieved]}")


Model:           qwen3.5:0.8b
Question:        What is the account code for Receivables for recoded but not paid in capital in 
GT:              {"account_number": "000", "name": "Receivables for recoded but not paid in capit
Duljina prompta: 1283 znakova
Procjena tokena: 320

Response:        000
Latency:         6.93s
EM:              True
Hallucin.:       False
Retrieved:       ['000', '002', '001', '066', '905']


In [11]:
from tqdm import tqdm

print("Predgeneriranje embeddings za sve upite...")
print("(preskače nomic — lagan, može u hodu)\n")

for embed_key in ["qwen3s", "qwen3m"]:
    print(f"  Model: {EMBED_MODELS[embed_key]}")
    new_count = 0
    for q in tqdm(test_set, desc=f"  {embed_key}", ncols=70):
        cache_key = f"{embed_key}|{q['question']}"
        if cache_key not in embed_cache:
            embed_cache[cache_key] = get_embedding(q["question"], embed_key)
            new_count += 1
    print(f"  Novo generirano: {new_count}")

save_embed_cache(embed_cache)
print(f"\n✓ Cache spreman: {len(embed_cache)} embeddings")
print("  Sigurno za pokretanje glavne petlje s 122b modelom.")

Predgeneriranje embeddings za sve upite...
(preskače nomic — lagan, može u hodu)

  Model: qwen3-embedding:0.6b


  qwen3s: 100%|███████████████████| 1060/1060 [01:11<00:00, 14.86it/s]


  Novo generirano: 1044
  Model: qwen3-embedding:4b


  qwen3m: 100%|███████████████████| 1060/1060 [01:09<00:00, 15.21it/s]


  Novo generirano: 1044

✓ Cache spreman: 2089 embeddings
  Sigurno za pokretanje glavne petlje s 122b modelom.


---
## 🚀 Main evaluation loop

In [12]:
print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
print(f"22 models × 5 conditions × 1,060 questions = 116,600 API calls")
print("=" * 65)

eval_start = time.time()

for model in ALL_MODELS:
    print(f"\n{'='*65}")
    print(f"MODEL: {model}  [{datetime.now().strftime('%H:%M:%S')}]")
    print(f"{'='*65}")

    for condition in CONDITIONS:
        if checkpoint_exists(model, condition):
            data     = load_checkpoint(model, condition)
            em_score = sum(r["exact_match"] for r in data) / len(data) * 100
            print(f"  ↩ {condition:20s} — skipped  (EM={em_score:.1f}%)")
            continue

        # Skip if ChromaDB collection missing for RAG conditions
        if condition != "no_rag" and loaded_colls.get(condition) is None:
            print(f"  ⚠ {condition:20s} — skipped  (collection missing)")
            continue

        print(f"\n  Running: {condition}  ({len(test_set)} questions)...")
        results = []
        errors  = 0
        t_cond  = time.time()

        for q in tqdm(test_set,
                      desc=f"  {model[:22]} [{condition[:12]}]",
                      ncols=72):
            retrieved = retrieve(q["question"], condition)
            prompt    = build_prompt(q["question"], retrieved, condition)
            ctx_ids   = [
                r["metadata"].get("account_number",
                r["metadata"].get("kljuc",""))
                for r in retrieved
            ]
            try:
                #response, latency = query_model(model, prompt)
                response, latency = query_model(model, prompt, q_type=q["q_type"])
                error = None
            except Exception as e:
                response, latency, error = "", 0.0, str(e)
                errors += 1

            results.append({
                "question":      q["question"],
                "q_type":        q["q_type"],
                "ground_truth":  q["answer_raw"],
                "response":      response,
                "latency_sec":   round(latency, 3),
                "exact_match":   exact_match(response, q["answer"]),
                "hallucination": hallucination_check(response),
                "retrieved_ids": ctx_ids,
                "condition":     condition,
                "model":         model,
                "error":         error,
            })

        elapsed  = time.time() - t_cond
        total    = len(results)
        em_pct   = sum(r["exact_match"]   for r in results) / total * 100
        hall_pct = sum(r["hallucination"] for r in results) / total * 100
        avg_lat  = sum(r["latency_sec"]   for r in results) / total
        lkp_n    = sum(1 for r in results if r["q_type"]=="lookup") or 1
        lkp_em   = sum(r["exact_match"] for r in results if r["q_type"]=="lookup")

        print(f"  ✓ {condition:20s} {elapsed:5.0f}s  "
              f"EM={em_pct:.1f}% (lkp={lkp_em/lkp_n*100:.1f}%)  "
              f"Hall={hall_pct:.1f}%  Lat={avg_lat:.2f}s  Err={errors}")
        save_checkpoint(model, condition, results)
        print(f"  💾 {checkpoint_path(model, condition)}")

total_elapsed = time.time() - eval_start
print(f"\n{'='*65}")
print(f"✓ Evaluation complete — {total_elapsed/3600:.1f}h total")
print(f"  Results: {RESULTS_DIR}/")
print(f"  Next: 05_analyze.ipynb")


Start: 16:25:56
22 models × 5 conditions × 1,060 questions = 116,600 API calls

MODEL: qwen3.5:0.8b  [16:25:56]
  ↩ no_rag               — skipped  (EM=50.0%)
  ↩ rag_a                — skipped  (EM=96.1%)
  ↩ rag_b_nomic          — skipped  (EM=61.5%)
  ↩ rag_b_qwen3s         — skipped  (EM=82.9%)
  ↩ rag_b_qwen3m         — skipped  (EM=84.9%)

MODEL: qwen3.5:2b  [16:25:56]
  ↩ no_rag               — skipped  (EM=50.0%)
  ↩ rag_a                — skipped  (EM=79.5%)
  ↩ rag_b_nomic          — skipped  (EM=60.3%)
  ↩ rag_b_qwen3s         — skipped  (EM=85.2%)
  ↩ rag_b_qwen3m         — skipped  (EM=86.9%)

MODEL: qwen3.5:4b  [16:25:56]
  ↩ no_rag               — skipped  (EM=50.1%)
  ↩ rag_a                — skipped  (EM=98.4%)
  ↩ rag_b_nomic          — skipped  (EM=62.5%)
  ↩ rag_b_qwen3s         — skipped  (EM=91.2%)
  ↩ rag_b_qwen3m         — skipped  (EM=91.9%)

MODEL: qwen3.5:9b  [16:25:56]
  ↩ no_rag               — skipped  (EM=50.3%)
  ↩ rag_a                — skipped  (EM=87.

  qwen3.5:122b [rag_b_nomic]: 100%|█| 1060/1060 [26:26<00:00,  1.50s/it]


  ✓ rag_b_nomic           1587s  EM=70.8% (lkp=41.5%)  Hall=45.5%  Lat=1.45s  Err=0
  💾 data/results/qwen3_5_122b_rag_b_nomic.json

  Running: rag_b_qwen3s  (1060 questions)...


  qwen3.5:122b [rag_b_qwen3s]: 100%|█| 1060/1060 [20:47<00:00,  1.18s/it


  ✓ rag_b_qwen3s          1248s  EM=94.3% (lkp=88.9%)  Hall=52.1%  Lat=1.17s  Err=0
  💾 data/results/qwen3_5_122b_rag_b_qwen3s.json

  Running: rag_b_qwen3m  (1060 questions)...


  qwen3.5:122b [rag_b_qwen3m]: 100%|█| 1060/1060 [20:18<00:00,  1.15s/it


... (147 outputs left out)

  deepseek-r1:70b [rag_b_qwen3s]: 100%|█| 1060/1060 [54:56<00:00,  3.11s


  ✓ rag_b_qwen3s          3296s  EM=90.6% (lkp=85.1%)  Hall=19.0%  Lat=3.11s  Err=0
  💾 data/results/deepseek-r1_70b_rag_b_qwen3s.json

  Running: rag_b_qwen3m  (1060 questions)...


  deepseek-r1:70b [rag_b_qwen3m]: 100%|█| 1060/1060 [52:24<00:00,  2.97s

  ✓ rag_b_qwen3m          3145s  EM=91.6% (lkp=85.5%)  Hall=12.7%  Lat=2.96s  Err=0
  💾 data/results/deepseek-r1_70b_rag_b_qwen3m.json

✓ Evaluation complete — 19.5h total
  Results: data/results/
  Next: 05_analyze.ipynb


In [ ]:
import requests, json

# Test s eksplicitnim think: False
r = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model":    "qwen3.5:0.8b",
        "messages": [{"role": "user", "content": "What is 2+2?"}],
        "stream":   False,
        "think":    False,
        "options":  {"temperature": 0, "num_predict": 512},
    },
    timeout=300,
)
data = r.json()
print(f"Latency:   {data.get('total_duration', 0) / 1e9:.2f}s")
print(f"Response:  {data['message']['content'][:200]}")
print(f"Has think tags: {'<think>' in data['message']['content']}")

In [ ]:
import requests, time

# Test 1: Koliko traje samo generiranje bez RAG
t0 = time.time()
r = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model":    "qwen3.5:0.8b",
        "messages": [{"role": "user", "content": "Say: yes"}],
        "stream":   False,
        "think":    False,
        "options":  {"temperature": 0, "num_predict": 10},
    },
    timeout=60,
)
elapsed = time.time() - t0
data = r.json()
print(f"Latency (num_predict=10):  {elapsed:.2f}s")
print(f"Response: {data['message']['content']}")
print(f"Load duration:   {data.get('load_duration', 0)/1e9:.2f}s")
print(f"Prompt eval:     {data.get('prompt_eval_duration', 0)/1e9:.2f}s")
print(f"Eval duration:   {data.get('eval_duration', 0)/1e9:.2f}s")

In [ ]:
#PAZI BRISANJE SVEGA!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
import glob, os

# Obriši sve checkpointe
for f in glob.glob("data/results/*.json"):
    os.remove(f)
    print(f"Obrisano: {f}")

print("✓ Checkpointi obrisani")

In [ ]:
# Direktni test s num_predict=64
import requests, time

t0 = time.time()
r = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model":    "qwen3.5:0.8b",
        "messages": [{"role": "user", "content": "What is the account code for Land? Answer with just the number."}],
        "stream":   False,
        "think":    False,
        "options":  {"temperature": 0, "num_predict": 64},
    },
    timeout=60,
)
elapsed = time.time() - t0
data = r.json()
print(f"Latency:         {elapsed:.2f}s")
print(f"Response:        {data['message']['content'][:100]}")
print(f"Load duration:   {data.get('load_duration',0)/1e9:.2f}s")
print(f"Prompt eval:     {data.get('prompt_eval_duration',0)/1e9:.2f}s")
print(f"Eval duration:   {data.get('eval_duration',0)/1e9:.2f}s")
print(f"Eval count:      {data.get('eval_count',0)} tokens")

In [ ]:
import subprocess

# Provjeri vidljivost GPU-a unutar Jupyter kontejnera
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else f"nvidia-smi nije dostupan: {result.stderr}")

# Provjeri CUDA
result2 = subprocess.run(["python3", "-c", "import torch; print(torch.cuda.is_available())"], 
                          capture_output=True, text=True)
print(f"CUDA dostupna: {result2.stdout.strip()}")

In [ ]:
import requests, time

t0 = time.time()
r = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model":    "qwen3.5:0.8b",
        "messages": [{"role": "user", "content": "What is the account code for Land?"}],
        "stream":   False,
        "think":    False,
        "options":  {"temperature": 0, "num_predict": 64},
    },
    timeout=60,
)
elapsed = time.time() - t0
data = r.json()
print(f"Latency:         {elapsed:.2f}s")
print(f"Response:        {data['message']['content'][:100]}")
print(f"Load duration:   {data.get('load_duration',0)/1e9:.2f}s")
print(f"Prompt eval:     {data.get('prompt_eval_duration',0)/1e9:.2f}s")
print(f"Eval duration:   {data.get('eval_duration',0)/1e9:.2f}s")
print(f"Eval count:      {data.get('eval_count',0)} tokens")

In [1]:
import os

to_delete = [
    "data/results/qwen3_5_122b_rag_b_nomic.json",
    "data/results/qwen3_5_122b_rag_b_qwen3s.json",
]

for f in to_delete:
    if os.path.exists(f):
        os.remove(f)
        print(f"✓ Obrisano: {f}")
    else:
        print(f"⚠ Ne postoji: {f}")

✓ Obrisano: data/results/qwen3_5_122b_rag_b_nomic.json
✓ Obrisano: data/results/qwen3_5_122b_rag_b_qwen3s.json
